In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout, Activation, Input
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [22]:
import numpy as np
import pandas as pd



np.random.seed(42)

n_samples = 500

area_m2 = np.random.randint(50, 251, n_samples)
bedrooms = np.random.randint(1, 6, n_samples)
bathrooms = np.random.randint(1, 4, n_samples)
age_years = np.random.randint(0, 31, n_samples)
distance_to_center_km = np.round(
    np.random.uniform(1, 20, n_samples), 1
)
floor = np.random.randint(1, 11, n_samples)


noise = np.random.normal(0, 30000, n_samples)

price = (
    50000
    + area_m2 * 3000
    + bedrooms * 25000
    + bathrooms * 15000
    - age_years * 2000
    - distance_to_center_km * 5000
    + floor * 3000
    + noise
)

price = np.maximum(price, 50000)

df = pd.DataFrame({
    "area_m2": area_m2,
    "bedrooms": bedrooms,
    "bathrooms": bathrooms,
    "age_years": age_years,
    "distance_to_center_km": distance_to_center_km,
    "floor": floor,
    "target": price.astype(int)
})

display(df.head())
print("Number of samples:", len(df))

,area_m2,bedrooms,bathrooms,age_years,distance_to_center_km,floor,target
0,152,1,1,17,1.6,7,514920
1,229,5,3,26,2.5,8,879481
2,142,5,1,9,8.5,3,546669
3,64,5,1,6,3.5,5,361730
4,156,3,1,16,11.8,1,504905


Number of samples: 500


In [23]:
FEATURES = [
    "area_m2",
    "bedrooms",
    "bathrooms",
    "age_years",
    "distance_to_center_km",
    "floor"
]

TARGET = "target"

In [24]:
X = df[FEATURES].values
y = df[TARGET].values

In [25]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)


In [26]:
y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train.reshape(-1, 1)
).ravel()

y_val_scaled = y_scaler.transform(
    y_val.reshape(-1, 1)
).ravel()

y_test_scaled = y_scaler.transform(
    y_test.reshape(-1, 1)
).ravel()

In [27]:
model = Sequential([

    Input(shape=(len(FEATURES),)),

    Dense(128, kernel_regularizer=l2(0.005)),
    BatchNormalization(),
    Activation("relu"),
    Dropout(0.3),

    Dense(64, kernel_regularizer=l2(0.005)),
    BatchNormalization(),
    Activation("relu"),
    Dropout(0.3),

    Dense(32, kernel_regularizer=l2(0.005)),
    BatchNormalization(),
    Activation("relu"),
    Dropout(0.3),

    
    Dense(1)
])

In [28]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

In [29]:
history = model.fit(
    X_train,
    y_train_scaled,         
    validation_data=(X_val, y_val_scaled), 
    epochs=150,
    batch_size=4,
    verbose=1
)

Epoch 1/150
88/88 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 2.2518 - mae: 0.9702 - val_loss: 6.2706 - val_mae: 2.0655
Epoch 2/150
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.6892 - mae: 0.7871 - val_loss: 1.6804 - val_mae: 0.8405
Epoch 3/150
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.7393 - mae: 0.8138 - val_loss: 0.8914 - val_mae: 0.3792
Epoch 4/150
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.5653 - mae: 0.7389 - val_loss: 0.8763 - val_mae: 0.3326
Epoch 5/150
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.4620 - mae: 0.7058 - val_loss: 0.7742 - val_mae: 0.2645
Epoch 6/150
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.3738 - mae: 0.6688 - val_loss: 0.7538 - val_mae: 0.2406
Epoch 7/150
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.3460 - mae: 0.6634 - val_loss: 0.7444 - val_mae: 0.2294
Epoch 8/150
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.2247 - mae: 0.6007 - val_loss: 0.7407 - val_mae: 0.2419
Epoch 9/150
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.277

In [30]:
pred_scaled = model.predict(X_test).reshape(-1)

pred = y_scaler.inverse_transform(
    pred_scaled.reshape(-1, 1)
).ravel()

print("Actual:", y_test)
print("Predicted:", pred)


mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)




print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)




result = pd.DataFrame({
    "Actual_Price": y_test,
    "Predicted_Price": pred
})

print("\nPredictions:")
display(result)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
Actual: [261355 491396 282589 533818 497303 725309 688715 218958 219343 486455
 516979 412726 829955 810986 407430 752395 757071 755515 182831 637139
 557391 824185 383496 374343 804064 422300 595773 271746 277681 606077
 621908 447516 844462 396282 256038 537107 714156 573276 418193 620048
 487657 418703 461714 653008 605175 723621 724002 783302 795626 617269
 478380 390962 569053 773687 801759 603070 457591 606546 725134 552327
 677569 578696 623869 477066 313097 847347 351298 383913 508385 304535
 863356 572882 574087 502512 394484]
Predicted: [325408.34 530200.06 356544.75 576859.25 552885.25 740838.2  744006.1
 337421.75 350238.97 503554.3  570341.6  462738.28 767002.2  760767.6
 457360.66 739049.44 782702.94 730240.06 302536.72 634088.4  595066.8
 782314.1  402849.06 422776.12 766202.9  471293.75 616607.3  353584.6
 352289.1  652753.94 586930.4  434448.53 771925.5  463854.38 349045.7
 591575.   668773.8  589160.44 484737.7  634805.   522732.5

,Actual_Price,Predicted_Price
0,261355,325408.34375
1,491396,530200.06250
2,282589,356544.75000
3,533818,576859.25000
4,497303,552885.25000
...,...,...
70,863356,770753.12500
71,572882,586919.87500
72,574087,640014.62500
73,502512,549238.06250
